In [ ]:
%pip install opencv-python-headless
%pip install torchvision --index-url https://download.pytorch.org/whl/cu124

In [ ]:
#!/usr/bin/env python
# coding: utf-8
"""
Frame extraction for VideoMAE-V2.
Samples N frames uniformly per video, applies VideoMAE processor normalization,
saves as [N, 3, 224, 224] float32 tensor → frames/train/{id_EXIST}.pt
"""

import torch
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm import tqdm
from transformers import VideoMAEImageProcessorFast
import cv2

# ── Config ────────────────────────────────────────────────────────────────────
VIDEO_DIR   = Path("./Dataset/test/videos/")
OUTPUT_DIR  = Path("./frames/test/")
JSON_PATH   = "./Dataset/test/EXIST2026_test_clean.json"
NUM_FRAMES  = 16          # match VideoMAE checkpoint default
MODEL_NAME  = "MCG-NJU/videomae-base"
# ─────────────────────────────────────────────────────────────────────────────

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

processor = VideoMAEImageProcessorFast.from_pretrained(MODEL_NAME)

df = pd.read_json(JSON_PATH).transpose()

def sample_frames_uniform(video_path: Path, n: int) -> torch.Tensor | None:
    """
    Load video with cv2, sample n frames uniformly.
    Returns [n, H, W, C] uint8 numpy → torch tensor.
    """
    try:
        cap = cv2.VideoCapture(str(video_path))
        total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        if total == 0:
            cap.release()
            return None
        indices = np.linspace(0, total - 1, n, dtype=int)
        frames = []
        for idx in indices:
            cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
            ret, frame = cap.read()
            if not ret:
                # fallback: duplicate last good frame
                frame = frames[-1] if frames else np.zeros((224, 224, 3), dtype=np.uint8)
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)  # cv2 reads BGR
            frames.append(frame)
        cap.release()
        # [n, H, W, C] uint8
        return np.stack(frames)
    except Exception as e:
        print(f"  [ERR] cv2 read failed: {e}")
        return None


def process_frames(frames_np: np.ndarray) -> torch.Tensor:
    """
    Apply VideoMAE processor normalization.
    Input:  [N, H, W, C] uint8 numpy
    Output: [N, C, 224, 224] float32 normalized
    """
    frames_list = [frames_np[i] for i in range(frames_np.shape[0])]
    processed = processor(
        images=frames_list,
        return_tensors="pt",
    )
    # pixel_values: [1, N, C, H, W] → squeeze → [N, C, H, W]
    pixel_values = processed["pixel_values"].squeeze(0)
    return pixel_values.float()


skipped = []

for _, row in tqdm(df.iterrows(), total=len(df), desc="Extracting frames"):
    item_id  = str(row["id_EXIST"])
    out_path = OUTPUT_DIR / f"{item_id}.pt"

    if out_path.exists():
        continue  # resume-safe

    video_path = VIDEO_DIR / str(row["path_video"]).split("/")[-1]
    if not video_path.exists():
        # try with just filename from path_video field
        video_path = VIDEO_DIR / Path(row["path_video"]).name
    if not video_path.exists():
        print(f"[WARN] Video not found: {item_id}")
        skipped.append(item_id)
        # save zeros so dataset doesn't crash
        torch.save(torch.zeros(NUM_FRAMES, 3, 224, 224), out_path)
        continue

    raw = sample_frames_uniform(video_path, NUM_FRAMES)
    if raw is None:
        print(f"[WARN] Could not read video: {item_id}")
        skipped.append(item_id)
        torch.save(torch.zeros(NUM_FRAMES, 3, 224, 224), out_path)
        continue

    tensor = process_frames(raw)               # [16, 3, 224, 224]
    torch.save(tensor, out_path)

print(f"\nDone. Skipped {len(skipped)} videos.")
if skipped:
    print("Skipped IDs:", skipped[:20])